# Actividad 6: Explorando transformers preentrenados con Hugging Face

**Curso:** Deep Learning  
**Profesor:** Gonzalo A. Ruz  
**Ayudante:** Anthony D. Cho

En esta actividad aplicaremos modelos transformer preentrenados a distintas tareas de procesamiento de lenguaje natural usando `pipeline()`.

Trabajaremos con ejemplos simples y guiados para observar cómo cambian las capacidades del modelo según la tarea.

## Objetivos de la actividad

Al finalizar esta actividad deberías poder:

- usar `pipeline()` para distintas tareas de NLP,
- interpretar salidas de modelos preentrenados,
- comparar distintos usos de transformers,
- reflexionar sobre la diferencia entre modelos clásicos y modelos modernos preentrenados.

## Instrucciones
- La actividad debe ser realizada por los grupos de trabajo
- Responda cada pregunta en las celdas correspondientes
- Justifique brevemente sus respuestas cuando se solicite
- Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo Actividad6_Tupper_Tudor_Gorosito_Acosta.ipynb
- Subir el archivo al link de entrega Actividad 6 en webcursos que será habilitado
- __Fecha de entrega:__ Idealmente al final del bloque 2 de la clase del 27 de abril 2026. Fecha límite de entrega 4 de mayo 2026

## Integrantes (RUT - Nombre y apellido):

- 13.257.556-8 - Ricardo Lopez
- 16.789.149-7 - Camilo Muñoz
- 13.307.082-6 - Álvaro Iriarte
- 25.608.509-7 - Ranse Vidal


## Instrucciones generales

- Completa las celdas indicadas.
- Lee con atención los comentarios y preguntas.
- Responde las preguntas breves en las celdas Markdown donde se indique.
- Si alguna salida cambia levemente, no te preocupes: eso puede ocurrir según la versión del modelo o del entorno.

In [110]:
!pip -q install -U transformers accelerate sentencepiece

import random
import numpy as np
import torch
from transformers import pipeline

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 0 if torch.cuda.is_available() else -1

print(f"Dispositivo: {'GPU' if device == 0 else 'CPU'}")
print(f"Semilla: {SEED}")

# Si Colab pide reiniciar el entorno después de instalar librerías,
# reiniciar y ejecutar todo nuevamente desde arriba.

Dispositivo: GPU
Semilla: 42


> **Importante:** si Colab pide reiniciar el entorno, hazlo antes de continuar.

## Parte 1: análisis de sentimiento

Comenzaremos con una tarea clásica de NLP: clasificar si un texto expresa un sentimiento positivo o negativo.

Completa la siguiente celda para crear un pipeline de `sentiment-analysis`.

In [111]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Ahora aplica el modelo a tres textos de ejemplo y observa las predicciones.

In [96]:
texts = [
    "I loved this product. It is easy to use and works very well.",
    "This was a disappointing purchase. The quality is poor.",
    "The product is acceptable, but I expected something better."
]

In [115]:
sentiment_results = sentiment_pipeline(texts)

positive_texts = [
    text for text, res in zip(texts, sentiment_results)
    if res["label"] == "POSITIVE"
]

most_uncertain = min(
    zip(texts, sentiment_results),
    key=lambda x: abs(x[1]["score"] - 0.5)
)

most_uncertain_text = most_uncertain[0]
most_uncertain_score = most_uncertain[1]["score"]

print("Resultados de sentimiento:")
for text_item, result in zip(texts, sentiment_results):
    print(f"- Texto: {text_item}")
    print(f"  Predicción: {result['label']} | Score: {result['score']:.4f}")

print("\nTextos positivos:")
for t in positive_texts:
    print(f"- {t}")

print("\nTexto con mayor incertidumbre relativa:")
print(f"- {most_uncertain_text}")
print(f"- Score: {most_uncertain_score:.4f}")

Resultados de sentimiento:
- Texto: I loved this product. It is easy to use and works very well.
  Predicción: POSITIVE | Score: 0.9999
- Texto: This was a disappointing purchase. The quality is poor.
  Predicción: NEGATIVE | Score: 0.9998
- Texto: The product is acceptable, but I expected something better.
  Predicción: NEGATIVE | Score: 0.9093

Textos positivos:
- I loved this product. It is easy to use and works very well.

Texto con mayor incertidumbre relativa:
- The product is acceptable, but I expected something better.
- Score: 0.9093


### Pregunta 1

Observa las predicciones del modelo.

**Escribe una breve respuesta:**
- ¿Qué textos fueron clasificados como positivos?
- ¿Cuál parece generar más incertidumbre según el score?

**Respuesta:**

El texto clasificado como positivo fue: “I loved this product. It is easy to use and works very well.”.

El texto con mayor incertidumbre relativa fue: “The product is acceptable, but I expected something better.”, con score aproximado de 0.9093. Aunque el score sigue siendo alto, fue el menor nivel de confianza entre los tres ejemplos, por eso se considera el caso más incierto dentro del conjunto.

In [124]:
my_text_sentiment = "The workshop was demanding but very rewarding, and I learned a lot."

custom_sentiment_text = my_text_sentiment
custom_sentiment_result = sentiment_pipeline(custom_sentiment_text)
custom_sentiment_label = custom_sentiment_result[0]["label"]
custom_sentiment_score = custom_sentiment_result[0]["score"]

print(f"Texto: {custom_sentiment_text}")
print(f"Predicción: {custom_sentiment_label} | Score: {custom_sentiment_score:.4f}")

Texto: The workshop was demanding but very rewarding, and I learned a lot.
Predicción: POSITIVE | Score: 0.9997


## Parte 2: zero-shot classification

Ahora usaremos un pipeline que permite clasificar un texto con etiquetas propuestas por nosotros, sin entrenar un clasificador específico para esas clases.

Completa la siguiente celda con el pipeline adecuado.

In [112]:
zero_shot_pipeline = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [116]:
text = "A balanced diet with vegetables, fruits, and whole grains can improve long-term health."

candidate_labels = ["food", "health", "travel", "business"]

In [117]:
zero_shot_result = zero_shot_pipeline(text, candidate_labels)

best_label = zero_shot_result["labels"][0]
best_score = zero_shot_result["scores"][0]

print("Texto evaluado:")
print(text)

print("\nRanking zero-shot:")
for label, score in zip(zero_shot_result["labels"], zero_shot_result["scores"]):
    print(f"- {label}: {score:.4f}")

print(f"\nMejor etiqueta: {best_label} ({best_score:.4f})")

Texto evaluado:
A balanced diet with vegetables, fruits, and whole grains can improve long-term health.

Ranking zero-shot:
- health: 0.5843
- food: 0.4079
- business: 0.0040
- travel: 0.0038

Mejor etiqueta: health (0.5843)


### Pregunta 2

Según la salida del modelo:

- ¿cuál fue la etiqueta más probable?
- ¿te parece razonable esa predicción?
- ¿qué ventaja tiene este enfoque frente a entrenar un clasificador desde cero para cada conjunto de etiquetas?

**Respuesta:**

La etiqueta más probable fue **health**, con score aproximado de **0.5843**. Es razonable porque el texto menciona explícitamente beneficios de largo plazo para la salud. También puede ser esperable que **food** quede cerca, porque el texto habla de dieta y alimentos.

La ventaja del enfoque zero-shot es que permite clasificar textos usando etiquetas nuevas propuestas por el usuario, sin entrenar desde cero un clasificador específico para esas clases.

## Parte 3: question answering

Ahora probaremos una tarea de pregunta-respuesta.

En este caso, el modelo recibe:

- una **pregunta**,
- un **contexto**,

e intenta encontrar la mejor respuesta dentro del texto entregado.

Completa la siguiente celda con el pipeline adecuado.

In [113]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=device
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [123]:
context = (
    "The Amazon rainforest is the largest tropical rainforest in the world. "
    "It spans several countries in South America, with most of it located in Brazil. "
    "The Amazon plays an important role in regulating the global climate and contains a great diversity of species."
)

question = "Which country contains most of the Amazon rainforest?"

In [118]:
qa_result = qa_pipeline(question=question, context=context)

qa_answer = qa_result["answer"]
qa_score = qa_result["score"]

print(f"Pregunta: {question}")
print(f"Respuesta QA: {qa_answer}")
print(f"Score QA: {qa_score:.4f}")

Pregunta: Which country contains most of the Amazon rainforest?
Respuesta QA: Brazil
Score QA: 0.9891


### Pregunta 3

Mira la salida del modelo.

- ¿Qué respuesta encontró?
- ¿Por qué esta tarea es distinta de `text-generation`?

**Respuesta:**

El modelo encontró **Brazil** como respuesta, con score aproximado de **0.9891**.

Esta tarea es distinta de `text-generation` porque question answering busca y extrae una respuesta desde un contexto entregado. En cambio, generación de texto produce contenido nuevo token a token, por lo que es más flexible, pero también menos controlada.

## Parte 4: prueba tus propios ejemplos

Ahora escribe:

- un texto propio para análisis de sentimiento,
- un texto propio para zero-shot classification,
- una pregunta propia para el contexto dado.

La idea es experimentar un poco con los modelos.

In [119]:
my_text_zero_shot = "The company announced a new electric bus fleet to reduce city emissions."
my_candidate_labels = ["technology", "environment", "sports", "politics"]

custom_zero_shot_text = my_text_zero_shot
custom_candidate_labels = my_candidate_labels

custom_zero_shot_result = zero_shot_pipeline(custom_zero_shot_text, custom_candidate_labels)
custom_zero_shot_best_label = custom_zero_shot_result["labels"][0]
custom_zero_shot_best_score = custom_zero_shot_result["scores"][0]

print(f"Texto: {custom_zero_shot_text}")
print("Ranking zero-shot:")
for label, score in zip(custom_zero_shot_result["labels"], custom_zero_shot_result["scores"]):
    print(f"- {label}: {score:.4f}")

print(f"Mejor etiqueta: {custom_zero_shot_best_label} ({custom_zero_shot_best_score:.4f})")

Texto: The company announced a new electric bus fleet to reduce city emissions.
Ranking zero-shot:
- technology: 0.6277
- environment: 0.3606
- sports: 0.0072
- politics: 0.0046
Mejor etiqueta: technology (0.6277)


In [120]:
my_question = "What is one global role of the Amazon rainforest?"

custom_question = my_question
custom_qa_result = qa_pipeline(question=custom_question, context=context)
custom_qa_answer = custom_qa_result["answer"]
custom_qa_score = custom_qa_result["score"]

print(f"Pregunta: {custom_question}")
print(f"Respuesta QA: {custom_qa_answer}")
print(f"Score QA: {custom_qa_score:.4f}")

Pregunta: What is one global role of the Amazon rainforest?
Respuesta QA: regulating the global climate
Score QA: 0.8611


### Pregunta 4

Después de probar tus propios ejemplos, responde brevemente:

- ¿qué tarea te pareció más intuitiva?
- ¿cuál te pareció más sorprendente?
- ¿en cuál confiarías más para una aplicación real?

**Respuesta:**

La tarea más intuitiva fue **sentiment analysis**, porque entrega una etiqueta directa de polaridad. En el ejemplo propio clasificó el texto como **POSITIVE** con score aproximado de **0.9997**.

La tarea más sorprendente fue **question answering**. En particular, el modelo logró responder **“regulating the global climate”** con score aproximado de **0.8611**, lo que muestra que puede recuperar información específica desde un contexto.

Para una aplicación real confiaría más en **question answering** cuando el contexto está controlado, porque la respuesta queda anclada a un texto verificable. Zero-shot es muy flexible, pero puede depender bastante de cómo se definan las etiquetas candidatas.

## Parte 5: generación de texto

Prueba una tarea de generación de texto.

Completa la siguiente celda con el pipeline adecuado.

In [114]:
text_generation_pipeline = pipeline(
    "text-generation",
    model="distilgpt2",
    device=device
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Use su propio texto para generar texto nuevo

In [121]:
prompt = "Transformers are useful in education because"

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

generated_result = text_generation_pipeline(
    prompt,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    pad_token_id=text_generation_pipeline.tokenizer.eos_token_id
)

generated_output = generated_result[0]["generated_text"].strip()

print("Texto generado:")
print(generated_output)

Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto generado:
Transformers are useful in education because they can be used to further enhance our learning.


### Pregunta 5

¿Qué diferencia observas entre:

- clasificar un texto,
- responder una pregunta sobre un contexto,
- generar texto nuevo?

**Respuesta:**

Clasificar un texto consiste en asignar una etiqueta a una entrada completa, como positivo/negativo o una categoría candidata.

Question answering recibe una pregunta y un contexto, y extrae una respuesta desde ese texto.

La generación de texto produce contenido nuevo a partir de un prompt. Es más flexible y creativa, pero también menos controlable, porque puede generar información que no estaba explícitamente en un contexto dado.

## Reflexión final

En la sesión 4 trabajamos con modelos secuenciales como LSTM.
En esta sesión trabajamos con transformers preentrenados.

### Pregunta 6

Escribe una reflexión breve comparando ambos enfoques.

Puedes apoyarte en ideas como:

- entrenamiento desde cero vs modelo preentrenado,
- recurrencia vs atención,
- flexibilidad de tareas,
- facilidad de uso con herramientas modernas.

**Respuesta:**

Los modelos LSTM y los Transformers representan dos formas distintas de trabajar con secuencias. Las LSTM procesan la información de manera recurrente, paso a paso, manteniendo un estado interno que resume lo visto anteriormente. Por eso son útiles para texto, audio o series de tiempo, pero pueden volverse más costosas y difíciles de entrenar cuando las dependencias son largas.

Los Transformers, en cambio, reemplazan la recurrencia por mecanismos de atención. Mediante self-attention, cada token puede relacionarse con otras posiciones de la secuencia, incluso lejanas, construyendo representaciones contextualizadas de forma más paralelizable.

Otra diferencia importante es el uso de preentrenamiento. En una LSTM típica del curso, normalmente se entrena un modelo para una tarea específica. En cambio, los Transformers modernos suelen venir preentrenados en grandes corpus y luego se reutilizan para distintas tareas mediante herramientas como `pipeline()` de Hugging Face. Esto permite resolver sentiment analysis, zero-shot classification, question answering y text generation con muy poco código.

En síntesis, la sesión 4 mostró el enfoque secuencial clásico con RNN/LSTM/GRU; la sesión 5 introdujo la idea de reutilizar conocimiento mediante modelos preentrenados; y la sesión 6 conecta ambas ideas con Transformers, atención y modelos modernos de lenguaje.

In [122]:
required_vars = [
    "SEED",
    "device",
    "sentiment_pipeline",
    "sentiment_results",
    "positive_texts",
    "most_uncertain_text",
    "most_uncertain_score",
    "zero_shot_pipeline",
    "text",
    "candidate_labels",
    "zero_shot_result",
    "best_label",
    "best_score",
    "qa_pipeline",
    "context",
    "question",
    "qa_result",
    "qa_answer",
    "qa_score",
    "custom_sentiment_text",
    "custom_sentiment_result",
    "custom_sentiment_label",
    "custom_sentiment_score",
    "custom_zero_shot_text",
    "custom_candidate_labels",
    "custom_zero_shot_result",
    "custom_zero_shot_best_label",
    "custom_zero_shot_best_score",
    "custom_question",
    "custom_qa_result",
    "custom_qa_answer",
    "custom_qa_score",
    "text_generation_pipeline",
    "prompt",
    "generated_result",
    "generated_output"
]

missing = [var for var in required_vars if var not in globals()]

if len(missing) == 0:
    print("✅ Validación completa: notebook reproducible, ejecutado correctamente y con variables principales disponibles.")
else:
    print("❌ Faltan variables:", missing)

✅ Validación completa: notebook reproducible, ejecutado correctamente y con variables principales disponibles.
